In [1]:
import wikipediaapi
from wikipediaapi import SearchWhat
import os
import pandas as pd
from dotenv import load_dotenv
import requests
import json

load_dotenv()
MAIL = os.getenv("MAIL")
USER_AGENT = f"{os.getenv("USER_AGENT")} ({MAIL})"

wiki = wikipediaapi.Wikipedia(
    user_agent=USER_AGENT,
    language="fr"
)

In [ ]:
results = wiki.search("Rayman Origins", limit=10)

for title, page in results.pages.items():
    print(title)

In [8]:
results = wiki.search(
    "Legend Zelda Breath Wild",
    what=SearchWhat.NEAR_MATCH,
    limit=10
)

for title, page in results.pages.items():
    print(title)

In [12]:

from functools import lru_cache
from datetime import datetime
import re


def obtenir_label_wikidata(qid):
    """Récupère le meilleur label disponible pour une entité Wikidata."""
    if not qid:
        return None

    params = {
        "action": "wbgetentities",
        "ids": qid,
        "props": "labels",
        "languages": "fr|en|mul",
        "format": "json",
    }

    try:
        data = get_with_retry(WIKIDATA_API, params).json()
        entity = data.get("entities", {}).get(qid, {})
        labels = entity.get("labels", {})

        # Priorité au français, puis à l'anglais
        for langue in ("fr", "en", "mul"):
            label = labels.get(langue, {}).get("value")
            if label:
                return label

        # Si aucun label fr/en, récupérer les autres langues
        params_fallback = {
            "action": "wbgetentities",
            "ids": qid,
            "props": "labels",
            "format": "json",
        }

        data = get_with_retry(
            WIKIDATA_API, params_fallback
        ).json()

        labels = data.get("entities", {}).get(qid, {}).get("labels", {})

        # Favoriser les langues européennes courantes
        for langue in ("es", "de", "it", "pt", "nl"):
            label = labels.get(langue, {}).get("value")
            if label:
                return label

        # Dernier recours
        if labels:
            return next(iter(labels.values())).get("value", qid)

    except Exception as e:
        print(f"Erreur récupération label {qid} : {e}")

    return qid


def valeur_snak(snak):
    """Extrait la valeur d'un snak Wikidata."""
    if not snak or snak.get("snaktype") != "value":
        return None

    datavalue = snak.get("datavalue", {})
    value = datavalue.get("value")

    if isinstance(value, dict):
        return value.get("id") or value.get("text")

    return value


def date_wikidata_vers_chaine(snak):
    """
    Convertit une date Wikidata en chaîne.
    Respecte la précision fournie par Wikidata.
    """
    if not snak or snak.get("snaktype") != "value":
        return None

    value = snak.get("datavalue", {}).get("value", {})

    if not isinstance(value, dict):
        return None

    date_brute = value.get("time")
    precision = value.get("precision", 11)

    if not date_brute:
        return None

    match = re.match(r"^[+-](\d+)-(\d{2})-(\d{2})", date_brute)
    if not match:
        return date_brute

    annee, mois, jour = map(int, match.groups())

    if precision >= 11:
        return f"{annee:04d}-{mois:02d}-{jour:02d}"
    elif precision == 10:
        return f"{annee:04d}-{mois:02d}"
    elif precision == 9:
        return f"{annee:04d}"

    return date_brute


def extraire_qualificatifs(claim):
    """
    Extrait les plateformes et les régions indiquées
    dans les qualificatifs de P577.
    """
    qualifiers = claim.get("qualifiers", {})

    plateformes = []
    regions = []

    # P400 = plateforme
    for snak in qualifiers.get("P400", []):
        qid = valeur_snak(snak)
        if qid:
            plateformes.append({
                "id": qid,
                "nom": obtenir_label_wikidata(qid),
            })

    # P291 = lieu de publication
    # P1001 = juridiction concernée
    # P3005 = applicable dans la région
    for prop in ("P291", "P1001", "P3005"):
        for snak in qualifiers.get(prop, []):
            qid = valeur_snak(snak)
            if qid:
                regions.append({
                    "propriete": prop,
                    "id": qid,
                    "nom": obtenir_label_wikidata(qid),
                })

    return plateformes, regions


def classer_region(regions):
    """
    Priorité :
    0 = France
    1 = Europe / Union européenne
    2 = aucune région indiquée
    3 = autre région indiquée
    """
    if not regions:
        return 2, "Non précisée"

    ids = {r["id"] for r in regions}
    noms = " ".join(r["nom"].lower() for r in regions)

    if "Q142" in ids or "france" in noms:
        return 0, "France"

    if (
        "Q46" in ids
        or "Q458" in ids
        or "europe" in noms
        or "union européenne" in noms
        or "union europeenne" in noms
    ):
        return 1, "Europe"

    return 3, ", ".join(r["nom"] for r in regions)


def extraire_dates_sortie(entite):
    """
    Extrait les dates de sortie depuis P577.

    Retourne les dates classées par priorité géographique,
    en conservant plateforme et région.
    """
    claims = entite.get("claims", {})
    dates = []

    for claim in claims.get("P577", []):
        mainsnak = claim.get("mainsnak", {})
        date = date_wikidata_vers_chaine(mainsnak)

        if not date:
            continue

        plateformes, regions = extraire_qualificatifs(claim)
        priorite, region_affichee = classer_region(regions)

        dates.append({
            "date": date,
            "plateformes": plateformes,
            "regions": regions,
            "region_prioritaire": region_affichee,
            "priorite_region": priorite,
        })

    # Les dates les plus pertinentes géographiquement en premier.
    dates.sort(key=lambda d: (d["priorite_region"], d["date"]))

    return dates


    
def extraire_entites_propriete(entite, propriete):
    """
    Extrait les entités Wikidata associées à une propriété.
    Exemples : P136 (genre), P178 (développeur), P123 (éditeur).
    """
    resultat = []
    claims = entite.get("claims", {})

    for claim in claims.get(propriete, []):
        snak = claim.get("mainsnak", {})
        qid = valeur_snak(snak)

        if not qid or not isinstance(qid, str) or not qid.startswith("Q"):
            continue

        resultat.append({
            "id": qid,
            "nom": obtenir_label_wikidata(qid),
            "wikidata_url": f"https://www.wikidata.org/wiki/{qid}",
        })

    # Supprime les doublons en conservant l'ordre.
    resultat_unique = []
    ids_vus = set()

    for item in resultat:
        if item["id"] not in ids_vus:
            resultat_unique.append(item)
            ids_vus.add(item["id"])

    return resultat_unique



In [19]:
WIKIDATA_API = "https://www.wikidata.org/w/api.php"
HEADERS = {"User-Agent": USER_AGENT}
import random
SPARQL_URL = "https://query.wikidata.org/sparql"
session = requests.Session()
session.headers.update(HEADERS)
def get_with_retry(url, params, max_retries=12):
    for tentative in range(max_retries):
        response = session.get(url, params=params, timeout=45)

        if response.status_code == 429:
            attente = response.headers.get("Retry-After")
            try:
                attente = float(attente)
            except (TypeError, ValueError):
                attente = min(60, 5 * (2 ** tentative))

            time.sleep(attente + random.uniform(0, 1))
            continue

        if response.status_code in (500, 502, 503, 504):
            time.sleep(min(60, 5 * (2 ** tentative)))
            continue

        response.raise_for_status()
        return response

    raise RuntimeError("Trop d'erreurs HTTP, réessaie plus tard.")

def est_un_jeu_video(entite):
    claims = entite.get("claims", {})

    if "P31" not in claims:
        return False

    for claim in claims["P31"]:
        try:
            qid = claim["mainsnak"]["datavalue"]["value"]["id"]

            if qid == "Q7889":
                return True

        except (KeyError, TypeError):
            pass

    return False

def rechercher_wikidata(titre, limit=10):
    """
    Recherche des entités Wikidata correspondant au titre.
    """

    params = {
        "action": "wbsearchentities",
        "search": titre,
        "language": "fr",
        "uselang": "fr",
        "limit": limit,
        "format": "json",
    }

    """response = requests.get(
        WIKIDATA_API, params=params, headers={"User-Agent": USER_AGENT}
    )"""

    response = get_with_retry(WIKIDATA_API, params, 12)

    #response.raise_for_status()

    return response.json()["search"]


def recuperer_entite(qid):
    """
    Récupère les données complètes d'une entité Wikidata.
    """

    params = {
        "action": "wbgetentities",
        "ids": qid,
        "format": "json",
        "props": "labels|claims|descriptions|sitelinks",
    }

    """response = requests.get(
        WIKIDATA_API, params=params, headers={"User-Agent": USER_AGENT}
    )

    response.raise_for_status()"""
    response = get_with_retry(WIKIDATA_API, params, 12)

    return response.json()["entities"][qid]


def extraire_ids(entite):
    """
    Extrait les identifiants RAWG et HowLongToBeat.
    """

    claims = entite.get("claims", {})

    resultat = {
        "wikidata_id": entite["id"],
        "rawg_id": None,
        "howlongtobeat_id": None,
    }

    # P9968 = RAWG game ID
    if "P9968" in claims:
        resultat["rawg_id"] = claims["P9968"][0]["mainsnak"]["datavalue"]["value"]

    # P2816 = HowLongToBeat ID
    if "P2816" in claims:
        resultat["howlongtobeat_id"] = claims["P2816"][0]["mainsnak"]["datavalue"][
            "value"
        ]

    return resultat


def trouver_jeu(titre):
    """
    Recherche un jeu et retourne les IDs disponibles.
    """

    candidats = rechercher_wikidata(titre)

    resultats = []

    for candidat in candidats:

        qid = candidat["id"]

        entite = recuperer_entite(qid)
        # On élimine tout ce qui n'est pas un jeu vidéo
        if not est_un_jeu_video(entite):
            continue

        ids = extraire_ids(entite)

        ids["label"] = candidat.get("label")
        ids["description"] = candidat.get("description")
        ids["dates_sortie"] = extraire_dates_sortie(entite)
        ids["genres"] = extraire_entites_propriete(entite, "P136")
        ids["developpeurs"] = extraire_entites_propriete(entite, "P178")
        ids["editeurs"] = extraire_entites_propriete(entite, "P123")
        ids["compositeurs"] = extraire_entites_propriete(entite, "P86")

        resultats.append(ids)

    return resultats


# Exemple
resultats = trouver_jeu("final fantasy")
print(f"Nombre des jeux Trouve:{len(resultats)}")
for jeu in resultats:

    print("---------------")
    print("Wikidata :", jeu["wikidata_id"])
    print("Nom      :", jeu["label"])
    print("Description :", jeu["description"])
    print("RAWG     :", jeu["rawg_id"])
    print("HLTB     :", jeu["howlongtobeat_id"])
    for champ in (
        "genres",
        "developpeurs",
        "editeurs",
        "plateformes",
        "compositeurs",
    ):
        print(f"{champ.capitalize()} :")

        for element in jeu.get(champ, []):
            print(f"  - {element['nom']} ({element['id']})")
    
    for sortie in jeu["dates_sortie"]:
        plateformes = ", ".join(
            p["nom"] for p in sortie["plateformes"]
        ) or "Plateforme non précisée"

        print(
            f"Date : {sortie['date']} | "
            f"Région : {sortie['region_prioritaire']} | "
            f"Plateforme : {plateformes}"
        )


Nombre des jeux Trouve:8
---------------
Wikidata : Q1415970
Nom      : Final Fantasy
Description : jeu vidéo de 1987
RAWG     : None
HLTB     : 3480
Genres :
  - jeu vidéo de rôle (Q744038)
Developpeurs :
  - Square (Q739552)
Editeurs :
  - Square (Q739552)
Plateformes :
Compositeurs :
  - Nobuo Uematsu (Q312327)
Date : 2010-05-07 | Région : Europe | Plateforme : Wii
Date : 1987-12-18 | Région : Japon | Plateforme : Nintendo Entertainment System
Date : 2009-05-26 | Région : Japon | Plateforme : Wii
Date : 2009-10-05 | Région : Amérique du Nord | Plateforme : Wii
---------------
Wikidata : Q214232
Nom      : Final Fantasy VII
Description : jeu vidéo de 1997
RAWG     : final-fantasy-vii
HLTB     : 3521
Genres :
  - jeu vidéo de rôle (Q744038)
  - jeu vidéo cyberpunk (Q54323360)
Developpeurs :
  - Square (Q739552)
Editeurs :
  - Square (Q739552)
  - Square Enix (Q207784)
Plateformes :
Compositeurs :
  - Nobuo Uematsu (Q312327)
Date : 1998-06-25 | Région : Europe | Plateforme : Microsoft 

In [ ]:

import requests
import time

SPARQL_URL = "https://query.wikidata.org/sparql"

HEADERS = {
    "User-Agent": USER_AGENT,
    "Accept": "application/sparql-results+json",
}

def get_jeux():


    
def recuperer_lots_jeux(qids):
    """Récupère les plateformes, dates et régions par lots."""

    qids = list(dict.fromkeys(q for q in qids if q.startswith("Q")))
    resultats = {}

    for debut in range(0, len(qids), 20):
        lot = qids[debut:debut + 20]
        valeurs = "".join(f"wd:{qid}" for qid in lot)

        query = f"""
        SELECT ?jeu ?jeuLabel
       ?plateforme ?plateformeLabel
       ?date
       ?genre ?genreLabel
       ?developpeur ?developpeurLabel
       ?publisher ?publisherLabel
       ?image ?hltb ?rawg
WHERE {{

    VALUES ?jeu {{ {valeurs} }}
    OPTIONAL {{
        ?jeu wdt:P2816 ?hltb .
    }}

    OPTIONAL {{
        ?jeu wdt:P2816 ?rawg .
    }}
    OPTIONAL {{
        ?jeu wdt:P400 ?plateforme .
    }}

    OPTIONAL {{
        ?jeu wdt:P577 ?date .
    }}

    OPTIONAL {{
        ?jeu wdt:P136 ?genre .
    }}

    OPTIONAL {{
        ?jeu wdt:P178 ?developpeur .
    }}

    OPTIONAL {{
        ?jeu wdt:P123 ?publisher .
    }}

    OPTIONAL {{
        ?jeu wdt:P18 ?image .
    }}

    SERVICE wikibase:label {{
        bd:serviceParam wikibase:language "fr,en" .
    }}
}}

        """

        response = requests.get(
            SPARQL_URL,
            params={"query": query},
            headers=HEADERS,
            timeout=60,
        )

        if response.status_code == 429:
            attente = int(response.headers.get("Retry-After", "10"))
            print(f"Limite atteinte, attente de {attente} secondes.")
            time.sleep(attente)
            response = requests.get(
                SPARQL_URL,
                params={"query": query},
                headers=HEADERS,
                timeout=60,
            )

        response.raise_for_status()
        lignes = response.json()["results"]["bindings"]

        for ligne in lignes:
            qid = ligne["jeu"]["value"].rsplit("/", 1)[-1]
            print(ligne)

            jeu = resultats.setdefault(qid, {
                "wikidata_id": qid,
                "nom": ligne.get("jeuLabel", {}).get("value"),
                "plateformes": {},
                "dates_sans_plateforme": [],
                "hltbid":ligne.get("hltb", {}).get("value"),
                "rawgid":ligne.get("rawg", {}).get("value"),
            })

            pid = ligne.get("plateforme", {}).get("value")
            if pid:
                pid = pid.rsplit("/", 1)[-1]
                plateforme = jeu["plateformes"].setdefault(pid, {
                    "id": pid,
                    "nom": ligne.get("plateformeLabel", {}).get("value"),
                    "sorties": [],
                })

                date = ligne.get("date", {}).get("value")
                if date:
                    sortie = {
                        "date": date,
                        "region": ligne.get("regionLabel", {}).get("value"),
                    }
                    if sortie not in plateforme["sorties"]:
                        plateforme["sorties"].append(sortie)

            elif ligne.get("date"):
                sortie = {
                    "date": ligne["date"]["value"],
                    "region": ligne.get("regionLabel", {}).get("value"),
                }
                if sortie not in jeu["dates_sans_plateforme"]:
                    jeu["dates_sans_plateforme"].append(sortie)

        # Pause entre les lots pour ménager le service.
        if debut + 20 < len(qids):
            time.sleep(2)

    for jeu in resultats.values():
        jeu["plateformes"] = list(jeu["plateformes"].values())

    return list(resultats.values())


# Exemple
jeux = recuperer_lots_jeux(["Q1070984"])

for jeu in jeux:
    print(jeu["nom"])
    print(jeu["plateformes"])
    print(jeu["hltbid"])
    print(jeu["rawgid"])


{'jeu': {'type': 'uri', 'value': 'http://www.wikidata.org/entity/Q1070984'}, 'hltb': {'type': 'literal', 'value': '8749'}, 'rawg': {'type': 'literal', 'value': '8749'}, 'plateforme': {'type': 'uri', 'value': 'http://www.wikidata.org/entity/Q1406'}, 'date': {'datatype': 'http://www.w3.org/2001/XMLSchema#dateTime', 'type': 'literal', 'value': '2001-06-19T00:00:00Z'}, 'genre': {'type': 'uri', 'value': 'http://www.wikidata.org/entity/Q828322'}, 'developpeur': {'type': 'uri', 'value': 'http://www.wikidata.org/entity/Q1140960'}, 'publisher': {'type': 'uri', 'value': 'http://www.wikidata.org/entity/Q122741'}, 'jeuLabel': {'xml:lang': 'fr', 'type': 'literal', 'value': 'Sonic Adventure 2'}, 'plateformeLabel': {'xml:lang': 'fr', 'type': 'literal', 'value': 'Microsoft Windows'}, 'genreLabel': {'xml:lang': 'fr', 'type': 'literal', 'value': 'jeu de plates-formes'}, 'developpeurLabel': {'xml:lang': 'fr', 'type': 'literal', 'value': 'Sonic Team'}, 'publisherLabel': {'xml:lang': 'fr', 'type': 'literal